## Imports

In [2]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, FunctionTransformer
from sklearn.impute import SimpleImputer

## Load Data

In [ ]:
df = pd.read_csv("../raw_data/hotel_bookings.csv")
df.head()

y = df["is_canceled"]
X = df.drop(columns="is_canceled")

train = df[df["arrival_date_year"] < 2017]
test = df[df["arrival_date_year"] == 2017]

X_train = train.drop(columns="is_canceled")
y_train = train["is_canceled"]

X_test = test.drop(columns="is_canceled")
y_test = test["is_canceled"]

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [6]:
from eda_package.registry import ORDINAL_FEATURES_MAP
from eda_package.feature_engineering import engineer_features
from eda_package.preprocessing import (
    get_included_countries,
    apply_country_grouping,
    get_feature_lists,
    create_preprocessor,
    fit_transform_preprocessor,
    transform_preprocessor
)

# 2. Feature engineering
X_train_fe = engineer_features(X_train)
X_test_fe = engineer_features(X_test)

# 3. Country grouping (fit on train)
included_countries = get_included_countries(X_train_fe, limit=100)

X_train_fe = apply_country_grouping(X_train_fe, included_countries)
X_test_fe = apply_country_grouping(X_test_fe, included_countries)

# 4. Feature lists (train only)
feature_lists = get_feature_lists(X_train_fe, ORDINAL_FEATURES_MAP)

# 5. Preprocessor
preprocessor = create_preprocessor(feature_lists, ORDINAL_FEATURES_MAP)

# 6. Transform
X_train_processed = fit_transform_preprocessor(X_train_fe, preprocessor)
X_test_processed = transform_preprocessor(X_test_fe, preprocessor)

ModuleNotFoundError: No module named 'eda_package'

In [ ]:
# add this, group country, and feature creation in features.py?

def engineer_features(X: pd.DataFrame) -> pd.DataFrame:
    """
    Create engineered features from the raw hotel booking dataset:
    - removes the target if it is present
    - removes known leakage columns
    - converts ID-like columns into binary indicators
    """
    X = X.copy()

    # Drop target if it is accidentally included
    if "is_canceled" in X.columns:
        X = X.drop(columns="is_canceled")

    # Drop leakage columns: these contain future information
    leakage_cols = ["reservation_status", "reservation_status_date"]
    cols_to_drop = [col for col in leakage_cols if col in X.columns]
    if cols_to_drop:
        X = X.drop(columns=cols_to_drop)

    # company ID -> binary flag
    if "company" in X.columns:
        X["has_company"] = (X["company"] != 0).astype(int)

    # agent ID -> binary flag
    if "agent" in X.columns:
        X["has_agent"] = (X["agent"] != 0).astype(int)

    # parking spaces -> simpler yes/no feature
    if "required_car_parking_spaces" in X.columns:
        X["has_parking"] = (X["required_car_parking_spaces"] > 0).astype(int)

    return X